<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset

## Imports

In [ ]:
import os

import torch

from torch.utils.data import Dataset
from torchvision.io import read_image

import matplotlib.pyplot as plt

import cv2

import numpy as np

import h5py

## SegmentationDatasetSeperateMasks

In [ ]:
class SegmentationDatasetSeperateMsks(Dataset):
  def __init__(self, img_dir, msk_dir, label_map, img_transform=None, msk_transform=None, msk_type=torch.float):
    self.img_dir = img_dir
    self.msk_dir = msk_dir
    self.ids = os.listdir(img_dir)
    self.label_map = label_map
    self.img_transform = img_transform
    self.msk_transform = msk_transform
    self.msk_type = msk_type

  def __len__(self):
    return len(self.ids)

  def __getitem__(self, idx):
    img_path = os.path.join(self.img_dir, self.ids[idx])
    msk_path = os.path.join(self.msk_dir, self.ids[idx])

    img = read_image(img_path)
    msk = read_image(msk_path)

    msks = [(msk == pixel_color) for pixel_color in self.label_map.values()]

    msk = torch.cat(msks, dim=0)

    if self.img_transform:
      img = self.img_transform(img)

    if self.msk_transform:
      msk = self.msk_transform(msk)

    return img, msk.type(self.msk_type)

## SegmentationDatasetJoinedMasks

In [ ]:
class SegmentationDatasetJoinedMsks(Dataset):
  def __init__(self, img_dir, msk_dir, img_transform=None, msk_transform=None, msk_type=torch.float):
    # Set the various attributes
    self.img_dir = img_dir
    self.msk_dir = msk_dir
    self.ids = os.listdir(img_dir) # Get's a list of names of all the files in img_dir (should only be images)
    self.img_transform = img_transform
    self.msk_transform = msk_transform
    self.msk_type = msk_type

  def __len__(self):
    return len(self.ids)

  def __getitem__(self, idx):
    # Get the image and mask path for the requested image
    img_path = os.path.join(self.img_dir, self.ids[idx])
    msk_path = os.path.join(self.msk_dir, self.ids[idx])

    # Read in the image and mask as a tensor
    img = read_image(img_path)
    msk = read_image(msk_path)

    # If there are any transforms, apply them respectively
    if self.img_transform:
      img = self.img_transform(img)

    if self.msk_transform:
      msk = self.msk_transform(msk)

    # Remove the unneccessary dimension
    msk = torch.squeeze(msk)

    return img, msk.to(self.msk_type)

## MultipleSegmentationDatasetsSeperateMsks

In [ ]:
class MultipleSegmentationDatasetsSeperateMsks(Dataset):
  def __init__(self, ds_map, lbl_map, joint_tr=None, img_post_tr=None, msk_post_tr=None):
    self.ds_map = ds_map
    self.lbl_map = lbl_map
    self.joint_tr = joint_tr
    self.img_post_tr = img_post_tr
    self.msk_post_tr = msk_post_tr
    self.dataset_idx_range_map = {}
    self.length = 0

    index_begin = 0
    for dataset_name, dataset_attrs in self.ds_map.items():
      ids = os.listdir(dataset_attrs["img_dir"])
      dataset_attrs["ids"] = ids
      self.dataset_idx_range_map[(index_begin, index_begin + len(ids))] = dataset_name
      index_begin += len(ids)

    self.length = index_begin

  def __len__(self):
    return self.length

  def __getitem__(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img_path = os.path.join(self.ds_map[src_dataset_name]["img_dir"], self.ds_map[src_dataset_name]["ids"][idx - idx_offset])
    msk_path = os.path.join(self.ds_map[src_dataset_name]["msk_dir"], self.ds_map[src_dataset_name]["img_to_msk_file_name"](self.ds_map[src_dataset_name]["ids"][idx - idx_offset]))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)

    msks = [(msk == pixel_color) for pixel_color in self.lbl_map.values()]
    msk = np.stack(msks, axis=-1).astype("float")

    if self.joint_tr:
      out_joint_tr = self.joint_tr(image=img, mask=msk)
      img, msk = out_joint_tr["image"], out_joint_tr["mask"]

    if self.img_post_tr:
      out_img_post_tr = self.img_post_tr(image=img)
      img = out_img_post_tr["image"]

    if self.msk_post_tr:
      out_msk_post_tr = self.msk_post_tr(image=msk, mask=msk)
      msk = out_msk_post_tr["mask"]

    return img, msk

  def get_unproc(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img_path = os.path.join(self.ds_map[src_dataset_name]["img_dir"], self.ds_map[src_dataset_name]["ids"][idx - idx_offset])
    msk_path = os.path.join(self.ds_map[src_dataset_name]["msk_dir"], self.ds_map[src_dataset_name]["img_to_msk_file_name"](self.ds_map[src_dataset_name]["ids"][idx - idx_offset]))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)

    return img, msk

In [ ]:
class MultipleSegmentationDatasetsSeperateMsks(Dataset):
  def __init__(self, ds_map, lbl_map, joint_tr=None, img_post_tr=None, msk_post_tr=None):
    self.ds_map = ds_map
    self.lbl_map = lbl_map
    self.joint_tr = joint_tr
    self.img_post_tr = img_post_tr
    self.msk_post_tr = msk_post_tr
    self.dataset_idx_range_map = {}
    self.length = 0

    index_begin = 0
    for dataset_name, dataset_attrs in self.ds_map.items():
      ids = os.listdir(dataset_attrs["img_dir"])
      dataset_attrs["ids"] = ids
      self.dataset_idx_range_map[(index_begin, index_begin + len(ids))] = dataset_name
      index_begin += len(ids)

    self.length = index_begin

  def __len__(self):
    return self.length

  def __getitem__(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img_path = os.path.join(self.ds_map[src_dataset_name]["img_dir"], self.ds_map[src_dataset_name]["ids"][idx - idx_offset])
    msk_path = os.path.join(self.ds_map[src_dataset_name]["msk_dir"], self.ds_map[src_dataset_name]["img_to_msk_file_name"](self.ds_map[src_dataset_name]["ids"][idx - idx_offset]))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)

    msks = [(msk == pixel_color) for pixel_color in self.lbl_map.values()]
    msk = np.stack(msks, axis=-1).astype("float")

    if self.joint_tr:
      out_joint_tr = self.joint_tr(image=img, mask=msk)
      img, msk = out_joint_tr["image"], out_joint_tr["mask"]

    if self.img_post_tr:
      out_img_post_tr = self.img_post_tr(image=img)
      img = out_img_post_tr["image"]

    if self.msk_post_tr:
      out_msk_post_tr = self.msk_post_tr(image=msk, mask=msk)
      msk = out_msk_post_tr["mask"]

    return img, msk

  def get_unproc(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img_path = os.path.join(self.ds_map[src_dataset_name]["img_dir"], self.ds_map[src_dataset_name]["ids"][idx - idx_offset])
    msk_path = os.path.join(self.ds_map[src_dataset_name]["msk_dir"], self.ds_map[src_dataset_name]["img_to_msk_file_name"](self.ds_map[src_dataset_name]["ids"][idx - idx_offset]))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)

    return img, msk

In [ ]:
class HDF5MultipleSegmentationDatasetsSeperateMsks(Dataset):
  def __init__(self, ds_map, lbl_map, joint_tr=None, img_post_tr=None, msk_post_tr=None):
    self.ds_map = ds_map
    self.lbl_map = lbl_map
    self.joint_tr = joint_tr
    self.img_post_tr = img_post_tr
    self.msk_post_tr = msk_post_tr
    self.dataset_idx_range_map = {}
    self.length = 0

    index_begin = 0
    for ds_name, ds_attr in self.ds_map.items():
      with h5py.File(ds_attr["h5_file_pth"], "r") as f:
        curr_sub_ds_len = len(f[ds_attr["sub_ds_name"]]["imgs"])
        self.dataset_idx_range_map[(index_begin, index_begin + curr_sub_ds_len)] = ds_name
        index_begin += curr_sub_ds_len

    self.length = index_begin

  def __len__(self):
    return self.length

  def __getitem__(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img = None
    msk = None

    with h5py.File(self.ds_map[src_dataset_name]["h5_file_pth"], "r") as f:
      img = f[self.ds_map[src_dataset_name]["sub_ds_name"]]["imgs"][idx - idx_offset]
      msk = f[self.ds_map[src_dataset_name]["sub_ds_name"]]["msks"][idx - idx_offset]

    msks = [np.isin(msk, pixel_values).astype("float") for pixel_values in self.lbl_map.values()]
    msk = np.stack(msks, axis=-1)

    if self.joint_tr:
      out_joint_tr = self.joint_tr(image=img, mask=msk)
      img, msk = out_joint_tr["image"], out_joint_tr["mask"]

    if self.img_post_tr:
      out_img_post_tr = self.img_post_tr(image=img)
      img = out_img_post_tr["image"]

    if self.msk_post_tr:
      out_msk_post_tr = self.msk_post_tr(image=msk, mask=msk)
      msk = out_msk_post_tr["mask"]

    return img, msk

  def get_unproc(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img = None
    msk = None

    with h5py.File(self.ds_map[src_dataset_name]["h5_file_pth"], "r") as f:
      img = f[self.ds_map[src_dataset_name]["sub_ds_name"]]["imgs"][idx - idx_offset]
      msk = f[self.ds_map[src_dataset_name]["sub_ds_name"]]["msks"][idx - idx_offset]

    msks = [np.isin(msk, pixel_values).astype("float") for pixel_values in self.lbl_map.values()]
    msk = np.stack(msks, axis=-1)

    return img, msk

  @classmethod
  def split_msk(cls, msk):
    msks = [np.isin(msk, pixel_values).astype("float") for pixel_values in self.lbl_map.values()]
    msk = np.stack(msks, axis=-1)

    return msk

In [ ]:
class HDF5MultipleSegmentationDatasetsJoinedMsks(Dataset):
  def __init__(self, ds_map, lbl_map, joint_tr=None, img_post_tr=None, msk_post_tr=None):
    self.ds_map = ds_map
    self.lbl_map = lbl_map
    self.joint_tr = joint_tr
    self.img_post_tr = img_post_tr
    self.msk_post_tr = msk_post_tr
    self.dataset_idx_range_map = {}
    self.length = 0

    index_begin = 0
    for ds_name, ds_attr in self.ds_map.items():
      with h5py.File(ds_attr["h5_file_pth"], "r") as f:
        curr_sub_ds_len = len(f[ds_attr["sub_ds_name"]]["imgs"])
        self.dataset_idx_range_map[(index_begin, index_begin + curr_sub_ds_len)] = ds_name
        index_begin += curr_sub_ds_len

    self.length = index_begin

  def __len__(self):
    return self.length

  def __getitem__(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img = None
    msk = None

    with h5py.File(self.ds_map[src_dataset_name]["h5_file_pth"], "r") as f:
      img = f[self.ds_map[src_dataset_name]["sub_ds_name"]]["imgs"][idx - idx_offset]
      msk = f[self.ds_map[src_dataset_name]["sub_ds_name"]]["msks"][idx - idx_offset]

    if self.joint_tr:
      out_joint_tr = self.joint_tr(image=img, mask=msk)
      img, msk = out_joint_tr["image"], out_joint_tr["mask"]

    if self.img_post_tr:
      out_img_post_tr = self.img_post_tr(image=img)
      img = out_img_post_tr["image"]

    if self.msk_post_tr:
      out_msk_post_tr = self.msk_post_tr(image=msk, mask=msk)
      msk = out_msk_post_tr["mask"]

    return img, msk

  def get_unproc(self, idx):
    src_dataset_name = None
    idx_offset = 0

    for dataset_index_range, dataset_name in self.dataset_idx_range_map.items():
      if (idx in range(*dataset_index_range)):
        src_dataset_name = dataset_name
        idx_offset = dataset_index_range[0]

    img_path = os.path.join(self.ds_map[src_dataset_name]["img_dir"], self.ds_map[src_dataset_name]["ids"][idx - idx_offset])
    msk_path = os.path.join(self.ds_map[src_dataset_name]["msk_dir"], self.ds_map[src_dataset_name]["img_to_msk_file_name"](self.ds_map[src_dataset_name]["ids"][idx - idx_offset]))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)

    return img, msk

## HDF5MultSeqSegDsSepMsks

In [ ]:
class HDF5MultSeqSegDsSepMsks(Dataset):
  def __init__(self, ds_map, lbl_map, is_out_tensor=True, joint_tr=None, img_post_tr=None, msk_post_tr=None):
    self.ds_map = ds_map
    self.lbl_map = lbl_map
    self.joint_tr = joint_tr
    self.img_post_tr = img_post_tr
    self.msk_post_tr = msk_post_tr
    self.seq_num_to_loc = []
    self.is_out_tensor = True

    for ds_name, ds_attrs in self.ds_map.items():
      with h5py.File(ds_attrs["h5_file_pth"], "r") as f:
        for act_seq_num in f[ds_attrs["sub_ds_name"]].keys():
          self.seq_num_to_loc.append({
              "ds_name": ds_name,
              "act_seq_num": act_seq_num
          })

  def __len__(self):
    return len(self.seq_num_to_loc)

  def __getitem__(self, idx):
    src_ds_name = self.seq_num_to_loc[idx]["ds_name"]
    act_seq_num = self.seq_num_to_loc[idx]["act_seq_num"]

    imgs = None
    msks = None

    with h5py.File(self.ds_map[src_ds_name]["h5_file_pth"], "r") as f:
      imgs_ds = f[self.ds_map[src_ds_name]["sub_ds_name"]][act_seq_num]["imgs"]

      if self.is_out_tensor:
        imgs = np.zeros((imgs_ds.shape[0], imgs_ds.shape[-1], imgs_ds.shape[-3], imgs_ds.shape[-2]), dtype=np.single)
      else:
        imgs = np.zeros(imgs_ds.shape, dtype=np.single)

      msks_ds = f[self.ds_map[src_ds_name]["sub_ds_name"]][act_seq_num]["msks"]

      if self.is_out_tensor:
        msks = np.zeros((msks_ds.shape[0], len(self.lbl_map), msks_ds.shape[-2], msks_ds.shape[-1]), dtype=int)
      else:
        msks = np.zeros(msks_ds.shape, dtype=int)

      for i in range(len(imgs_ds)):
        img = imgs_ds[i]
        msk = msks_ds[i]

        msk = np.stack([np.isin(msk, pixel_values).astype("float") for pixel_values in self.lbl_map.values()], axis=-1)

        if self.joint_tr:
          out_joint_tr = self.joint_tr(image=img, mask=msk)
          img, msk = out_joint_tr["image"], out_joint_tr["mask"]

        if self.img_post_tr:
          out_img_post_tr = self.img_post_tr(image=img)
          img = out_img_post_tr["image"]

        if self.msk_post_tr:
          out_msk_post_tr = self.msk_post_tr(image=msk, mask=msk)
          msk = out_msk_post_tr["mask"]
        # print(np.unique(img))

        imgs[i] = img
        msks[i] = msk



    return act_seq_num, imgs, msks

## Testing

In [ ]:
# import albumentations as A

In [ ]:
# !cp "/content/drive/My Drive/3yp/datasets/kitti_step.h5" .

In [ ]:
# def to_tensor(x, cols, rows):
#   return x.transpose(2, 0, 1)

In [ ]:
# img_tr = A.Compose(
#     [
#       A.Normalize(mean=0.0, std=1.0), # 0/1 Normalization
#       A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=1), # ImageNet Normalization,
#       A.Lambda(image=to_tensor)
#     ]
# )

In [ ]:
# msk_tr = A.Compose([
#   A.Lambda(image=to_tensor, mask=to_tensor)
# ])

In [ ]:
# train_ds = HDF5MultSeqSegDsSepMsks(
#     ds_map={
#         "gta_train": {
#             "h5_file_pth": "./kitti_step.h5",
#             "sub_ds_name": "train"
#         },
#     },
#     lbl_map={"some_class": 0, "class_2": 1},
#     img_post_tr=img_tr,
#     msk_post_tr=msk_tr
# )


(154, 2, 384, 384)

In [ ]:
# !cp "/content/drive/My Drive/3yp/datasets/gta.h5" .

In [ ]:
# Example code
# bench_MultipleSegmentationDatasetsSeperateMasks = MultipleSegmentationDatasetsSeperateMasks(
#     dataset_map={
#         "gta": {
#             "img_dir": "/content/drive/My Drive/datasets/gta/train/imgs",
#             "msk_dir": "/content/drive/My Drive/datasets/gta/train/msks",
#             "img_to_msk_file_name": lambda x: x,
#         },
#         "cityscapes": {
#             "img_dir": "/content/drive/My Drive/datasets/cityscapes/train/imgs",
#             "msk_dir": "/content/drive/My Drive/datasets/cityscapes/train/msks",
#             "img_to_msk_file_name": lambda x: f"{x[:-15]}gtFine_labelIds.png",
#         }
#     },
#     label_map={"some_class": 0, "class_2": 1}
# )

# bench_MultipleSegmentationDatasetsSeperateMasks = HDF5MultipleSegmentationDatasetsSeperateMsks(
#     ds_map={
#         "gta_train": {
#             "h5_file_pth": "./gta.h5",
#             "sub_ds_name": "train"
#         },
#     },
#     lbl_map={"some_class": 0, "class_2": 1}
# )

In [ ]:
# bench_MultipleSegmentationDatasetsSeperateMasks = HDF5MultipleSegmentationDatasetsJoinedMsks(
#     ds_map={
#         "gta_train": {
#             "h5_file_pth": "./gta.h5",
#             "sub_ds_name": "train"
#         },
#     },
#     lbl_map={"some_class": 0, "class_2": 1}
# )